# Titres français des films

Notre liste de films (`films_filtres.csv`, préparée dans le notebook précédent) contient les **titres originaux**, souvent en anglais. Pour un cinéma en France, nous voulons afficher le **titre sous lequel le film est sorti en France**.

Nous complétons aussi chaque film avec deux informations de **TMDB** :
- sa **langue originale** (`original_language`) ;
- son **affiche** (`poster_path`), utile plus tard pour l'application de recommandation.

## Démarche
1. **Charger** le fichier IMDb `title.akas`, qui liste tous les titres de chaque film dans chaque pays.
2. **Garder** seulement les titres français (`region == "FR"`) de nos films.
3. **Nettoyer** les doublons (un film peut avoir plusieurs titres en France).
4. **Fusionner** ces titres avec notre liste de films.
5. **Ajouter** la langue originale et l'affiche depuis TMDB.
6. **Examiner** les films qui n'ont pas de titre français, puis décider lesquels garder.
7. **Compléter** les titres manquants et **enregistrer** la sélection.


In [1]:
import pandas as pd

lien = "https://datasets.imdbws.com/title.akas.tsv.gz"
akas = pd.read_csv(lien, sep="\t", na_values="\\N")

C:\Users\bourr\AppData\Local\Temp\ipykernel_5736\3375504777.py:4: DtypeWarning: Columns (6: attributes) have mixed types. Specify dtype option on import or set low_memory=False.
  akas = pd.read_csv(lien, sep="\t", na_values="\\N")


Découvrir le fichier `title.akas` d'IMDb. « aka » veut dire *also known as* (« aussi connu sous le nom de ») : il contient **tous les titres d'un film, pays par pays**.

**Explication du code :**
- `import pandas as pd` charge **pandas**, la bibliothèque qui manipule des tableaux de données.
- `pd.read_csv(lien, sep="\t", na_values="\\N")` lit le fichier directement depuis Internet. Les colonnes sont séparées par des **tabulations** (`\t`), et IMDb écrit `\N` pour une valeur manquante, que pandas transforme en case vide.

In [2]:
akas.head()

,titleId,ordering,title,region,language,types,attributes,isOriginalTitle
0,tt0000001,1,Carmencita,NaN,NaN,original,NaN,1
1,tt0000001,2,Carmencita,DE,NaN,NaN,literal title,0
2,tt0000001,3,Carmencita,US,NaN,imdbDisplay,NaN,0
3,tt0000001,4,Carmencita - spanyol tánc,HU,NaN,imdbDisplay,NaN,0
4,tt0000001,5,Καρμενσίτα,GR,NaN,imdbDisplay,NaN,0


Vérifier le nom des colonnes et à quoi ressemblent les données avant de travailler dessus.

**Explication du code :**
- `akas.head()` affiche les 5 premières lignes du tableau.

Les colonnes utiles pour nous sont :
- `titleId` : l'identifiant IMDb du film (le même que `tconst` dans notre liste) ;
- `title` : le titre dans un pays donné ;
- `region` : le pays (`FR` pour la France, `US` pour les États-Unis...).

Nous voyons qu'un même film (`tt0000001`) apparaît **plusieurs fois**, une ligne par pays.

In [3]:
films = pd.read_csv("../data/films_filtres.csv")
films.head()

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0035423,movie,Kate & Leopold,0,2001.0,118.0,"Comedy,Fantasy,Romance",6.4,95169
1,tt0064106,movie,The Butcher,0,1970.0,93.0,"Drama,Mystery,Thriller",7.3,10675
2,tt0064177,movie,Colossus: The Forbin Project,0,1970.0,100.0,"Sci-Fi,Thriller",7.1,12058
3,tt0065134,movie,Two Mules for Sister Sara,0,1970.0,116.0,"Adventure,Drama,Romance",7.0,33505
4,tt0065143,movie,The Bird with the Crystal Plumage,0,1970.0,96.0,"Drama,Horror,Mystery",7.1,27767


Récupérer la liste des films sélectionnés dans le notebook précédent.

**Explication du code :**
- `pd.read_csv("../data/films_filtres.csv")` lit le fichier CSV. `../` veut dire « remonter d'un dossier » : nous partons du dossier `notebooks/` pour aller dans `data/`.
- `films.head()` affiche les 5 premières lignes pour vérifier.

Nous retrouvons nos colonnes habituelles : identifiant, titre original (`primaryTitle`), année, durée, genres, note et nombre de votes.

In [4]:
colonnes = ['titleId', 'title', 'region']

nos_films = set(films['tconst'])

akas = pd.read_csv(
    lien,
    sep="\t",
    na_values="\\N",
    usecols=colonnes,
    chunksize=2_000_000,
    )

titres_gardes = []
for morceau in akas:
    est_fr = morceau["region"] == "FR"
    est_a_nous = morceau["titleId"].isin(nos_films)
    titres_gardes.append(morceau[est_fr & est_a_nous])

akas_fr = pd.concat(titres_gardes, ignore_index=True).drop_duplicates()

Garder uniquement les titres français de nos films. Le fichier `akas` est énorme (des dizaines de millions de lignes) : le charger en entier puis filtrer prend beaucoup de mémoire. Nous le relisons donc par morceaux, et dans chaque morceau nous ne gardons que ce qui nous intéresse.

**Explication du code :**
- `colonnes = [...]` : nous ne lisons que 3 colonnes (identifiant, titre, pays), pour économiser la mémoire.
- `nos_films = set(films["tconst"])` met nos identifiants dans un **ensemble** (`set`). Chercher dedans est beaucoup plus rapide que dans une liste.
- `chunksize=2_000_000` : au lieu de renvoyer un tableau, `read_csv` renvoie des **morceaux de 2 millions de lignes**, lus l'un après l'autre. (`2_000_000` s'écrit avec des `_` pour être plus lisible ; c'est le même nombre que 2000000.)
- La boucle `for morceau in akas:` traite chaque morceau :
  - `est_fr` vaut Vrai pour les lignes dont le pays est la **France** ;
  - `est_a_nous` vaut Vrai pour les lignes qui concernent **un de nos films** ;
  - `morceau[est_fr & est_a_nous]` garde les lignes où **les deux** sont vraies (`&` veut dire « et »), et nous les ajoutons à la liste `titres_gardes`.
- `pd.concat(titres_gardes, ignore_index=True)` **recolle** tous les petits morceaux en un seul tableau, avec des numéros de ligne propres.
- `.drop_duplicates()` retire les lignes **strictement identiques**.

Nous obtenons `akas_fr` : les titres français de nos films.

In [5]:
akas_fr.head()

,titleId,title,region
0,tt0035423,Kate et Léopold,FR
1,tt0064106,Le boucher,FR
2,tt0064177,Le cerveau d'acier,FR
3,tt0065134,Sierra torride,FR
4,tt0065143,L'oiseau au plumage de cristal,FR


Contrôler que le filtre a bien marché.

**Explication du code :**
- `akas_fr.head()` affiche les 5 premières lignes du tableau filtré.

Nous voyons bien des titres français (*Kate et Léopold*, *Le boucher*, *Sierra torride*...) et la colonne `region` vaut `FR` partout.

In [6]:
akas_fr.shape

(6589, 3)

Comparer le nombre de titres français avec le nombre de films de notre liste.

**Explication du code :**
- `.shape` donne (nombre de lignes, nombre de colonnes).

`akas_fr` contient **6 589 lignes**, alors que notre liste n'a que **6 447 films**. Il y a donc **plus de lignes que de films** : certains films ont plusieurs titres français. Il faut les dédoublonner.

In [7]:
akas_fr.duplicated("titleId").sum()

np.int64(608)

Savoir combien de films apparaissent plusieurs fois.

**Explication du code :**
- `duplicated("titleId")` renvoie Vrai pour chaque ligne dont l'identifiant a **déjà été vu plus haut**.
- `.sum()` compte ces Vrai.

**608 lignes** sont des doublons : un même film avec plusieurs titres en France (par exemple un titre de sortie en salle et un titre de sortie vidéo).

In [8]:
akas_fr = akas_fr.drop_duplicates(subset="titleId")

Garder **un seul titre français par film**.

**Explication du code :**
- `drop_duplicates(subset="titleId")` ne regarde que la colonne `titleId` et garde la **première ligne** de chaque film. Les suivantes sont supprimées.

**Point d'attention.** Nous gardons le premier titre trouvé, sans choisir lequel est le « meilleur ». C'est un choix simple, à affiner si besoin.

In [9]:
akas_fr.shape

(5981, 3)

Vérifier le nombre de films après dédoublonnage.

**Explication du code :**
- `akas_fr.shape` donne à nouveau (nombre de lignes, nombre de colonnes).

Il reste **5 981 films** avec un titre français (6 589 − 608). Cela veut dire que **466 films** de notre liste (6 447 − 5 981) n'ont **pas de titre français** dans IMDb.

In [10]:
akas_fr[akas_fr.duplicated("title")]

,titleId,title,region
807,tt0091223,House,FR
1114,tt0099012,Alice,FR
1293,tt0103874,Dracula,FR
1446,tt0108333,Les trois mousquetaires,FR
2055,tt0149261,Peur bleue,FR
...,...,...,...
6204,tt6793580,Champions,FR
6300,tt7390588,Little Monsters,FR
6399,tt8097030,Alerte rouge,FR
6493,tt8946378,À couteaux tirés,FR


Vérifier si plusieurs films **différents** portent le **même titre**.

**Explication du code :**
- `duplicated("title")` renvoie Vrai quand un titre a déjà été vu.
- `akas_fr[...]` affiche ces lignes.

**76 films** ont le même titre qu'un autre film (*House*, *Dracula*, *Les trois mousquetaires*...). Ce n'est **pas une erreur** : ce sont des remakes ou des films sans rapport. C'est pour cela que nous travaillons toujours avec l'**identifiant** (`tconst`) et jamais avec le titre.

In [11]:
films_francais = pd.merge(
    left=films,
    right=akas_fr,
    how="left",
    left_on="tconst",
    right_on="titleId"
).drop(columns=["titleId", "region"])

films_francais.head()

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title
0,tt0035423,movie,Kate & Leopold,0,2001.0,118.0,"Comedy,Fantasy,Romance",6.4,95169,Kate et Léopold
1,tt0064106,movie,The Butcher,0,1970.0,93.0,"Drama,Mystery,Thriller",7.3,10675,Le boucher
2,tt0064177,movie,Colossus: The Forbin Project,0,1970.0,100.0,"Sci-Fi,Thriller",7.1,12058,Le cerveau d'acier
3,tt0065134,movie,Two Mules for Sister Sara,0,1970.0,116.0,"Adventure,Drama,Romance",7.0,33505,Sierra torride
4,tt0065143,movie,The Bird with the Crystal Plumage,0,1970.0,96.0,"Drama,Horror,Mystery",7.1,27767,L'oiseau au plumage de cristal


Mettre le titre français **à côté de chaque film** de notre liste.

**Explication du code :**
- `pd.merge(...)` **fusionne** deux tableaux :
  - `left=films` : notre liste de films (tableau de gauche) ;
  - `right=akas_fr` : les titres français (tableau de droite) ;
  - `left_on="tconst"` et `right_on="titleId"` : les colonnes qui servent de **clé** pour relier les deux tableaux. Elles n'ont pas le même nom, mais contiennent le même identifiant.
  - `how="left"` : nous gardons **tous les films de gauche**, même ceux qui n'ont pas de titre français. Pour eux, la colonne `title` sera vide.
- `.drop(columns=["titleId", "region"])` supprime les colonnes devenues inutiles : `titleId` répète `tconst`, et `region` vaut toujours `FR`.
- `films_francais.head()` affiche les 5 premières lignes.

Notre liste a maintenant une colonne `title` avec le titre français.

In [12]:
films_francais.shape

(6447, 10)

Vérifier que la fusion n'a pas changé le nombre de films.

**Explication du code :**
- `films_francais.shape` donne (nombre de lignes, nombre de colonnes).

Nous avons toujours **6 447 films** : grâce à `how="left"` et au dédoublonnage, la fusion n'a ni supprimé ni dupliqué de films. Il y a maintenant **10 colonnes**.

In [13]:
films_francais[films_francais["title"].isna()]

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title
164,tt0072783,movie,Chupke Chupke,0,1975.0,146.0,"Comedy,Drama,Romance",8.3,14189,NaN
270,tt0077451,movie,Don,0,1978.0,166.0,"Action,Crime,Thriller",7.7,13685,NaN
310,tt0079221,movie,Gol Maal,0,1979.0,120.0,"Comedy,Romance",8.5,22413,NaN
377,tt0081590,movie,The Charter Trip,0,1980.0,107.0,Comedy,7.2,10096,NaN
380,tt0081748,movie,Where the Buffalo Roam,0,1980.0,99.0,"Biography,Comedy",6.5,13409,NaN
...,...,...,...,...,...,...,...,...,...,...
6366,tt9098938,movie,Sharmaji Namkeen,0,2022.0,119.0,"Comedy,Drama,Family",7.4,13915,NaN
6381,tt9248940,movie,Dream Girl,0,2019.0,132.0,"Comedy,Romance",7.0,18997,NaN
6397,tt9420648,movie,Bala,0,2019.0,133.0,Comedy,7.3,18495,NaN
6404,tt9537292,movie,Ala Vaikunthapurramuloo,0,2020.0,163.0,"Action,Comedy,Drama",7.3,27426,NaN


Regarder **quels films** n'ont pas de titre français.

**Explication du code :**
- `.isna()` renvoie Vrai quand une case est **vide**.
- `films_francais[...]` garde les lignes où `title` est vide.

**466 films** n'ont pas de titre français. Beaucoup sont des **films indiens** (*Chupke Chupke*, *Don*, *Gol Maal*...), qui ne sont sans doute jamais sortis en France.

In [14]:
films_francais[films_francais["primaryTitle"].str.contains("fight", case=False)]

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title
553,tt0087597,movie,The Last Starfighter,0,1984.0,100.0,"Action,Adventure,Sci-Fi",6.7,50611,Starfighter
1031,tt0101748,movie,Dogfight,0,1991.0,94.0,"Comedy,Drama,Romance",7.3,11514,L'amour n'est pas un jeu
1414,tt0114563,movie,Street Fighter II: The Animated Movie,0,1994.0,102.0,"Action,Adventure,Animation",7.2,11768,Street Fighter II: Le film
1768,tt0137523,movie,Fight Club,0,1999.0,139.0,"Crime,Drama,Thriller",8.8,2683106,Fight Club
2743,tt0436971,movie,Why We Fight,0,2005.0,98.0,"Documentary,History,War",8.0,10557,Le Nerf de la guerre
3188,tt0964517,movie,The Fighter,0,2010.0,116.0,"Action,Biography,Drama",7.8,412630,Fighter
3488,tt11456054,movie,Run Hide Fight,0,2020.0,109.0,"Action,Crime,Drama",6.3,31278,Run Hide Fight
3861,tt13652286,movie,Fight or Flight,0,2024.0,102.0,"Action,Comedy",6.4,29377,Fight or Flight
3881,tt13818368,movie,Fighter,0,2024.0,166.0,"Action,Drama",6.1,54832,Fighter
4936,tt2495118,movie,Ip Man: The Final Fight,0,2013.0,100.0,"Action,Biography,Drama",6.1,11928,Ip Man: Le combat final


Contrôler, sur des titres faciles à reconnaître, que les titres français sont cohérents.

**Explication du code :**
- `str.contains("fight", case=False)` garde les films dont le titre original contient le mot « fight ».
- `case=False` : la recherche se fait **sans tenir compte des majuscules**.

Nous comparons le titre original et le titre français : par exemple *Fighting with My Family* devient *Une famille sur le ring*, et *Ip Man: The Final Fight* devient *Ip Man: Le combat final*. Certains films gardent leur titre anglais en France (*Fight Club*), ce qui est normal.

In [15]:
films_tmdb = pd.read_csv("../data/TMDB_movie_dataset_v11.csv", usecols=["imdb_id", "original_language", "poster_path"])

films_tmdb.head()

,imdb_id,original_language,poster_path
0,tt1375666,en,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg
1,tt0816692,en,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg
2,tt0468569,en,/qJ2tW6WMUDux911r6m7haRef0WH.jpg
3,tt0499549,en,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg
4,tt0848228,en,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg


Récupérer dans TMDB deux informations qu'IMDb n'a pas : la **langue originale** du film et son **affiche**.

**Explication du code :**
- `pd.read_csv("../data/TMDB_movie_dataset_v11.csv", ...)` lit le fichier TMDB.
- `usecols=["imdb_id", "original_language", "poster_path"]` ne lit que 3 colonnes, car le fichier TMDB est très lourd (environ 670 Mo).
- `imdb_id` est l'identifiant IMDb : il va servir de **clé** pour relier TMDB à notre liste.
- `poster_path` est la fin de l'adresse de l'**image de l'affiche** (par exemple `/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg`).
- `original_language` est un code de langue : `en` pour anglais, `fr` pour français, `it` pour italien...

In [16]:
films_francais = pd.merge(
    left=films_francais,
    right=films_tmdb,
    how="left",
    left_on="tconst",
    right_on="imdb_id"
).drop(columns=["imdb_id"])

films_francais.head()

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path
0,tt0035423,movie,Kate & Leopold,0,2001.0,118.0,"Comedy,Fantasy,Romance",6.4,95169,Kate et Léopold,en,/mUvikzKJJSg9khrVdxK8kg3TMHA.jpg
1,tt0064106,movie,The Butcher,0,1970.0,93.0,"Drama,Mystery,Thriller",7.3,10675,Le boucher,fr,/An2gkXn5TATag9Adul2Zbq5Ndrv.jpg
2,tt0064177,movie,Colossus: The Forbin Project,0,1970.0,100.0,"Sci-Fi,Thriller",7.1,12058,Le cerveau d'acier,en,/kmVDlTzERxSvBrqtQurjKf1zEy1.jpg
3,tt0065134,movie,Two Mules for Sister Sara,0,1970.0,116.0,"Adventure,Drama,Romance",7.0,33505,Sierra torride,en,/RbuNHMlrw7M2w1R3hZMAPwh6dG.jpg
4,tt0065143,movie,The Bird with the Crystal Plumage,0,1970.0,96.0,"Drama,Horror,Mystery",7.1,27767,L'oiseau au plumage de cristal,it,/dpALRkwodEIT7TGX9SqTgwVNn0i.jpg


Mettre la langue originale et l'affiche **à côté de chaque film**.

**Explication du code :**
- `pd.merge(...)` fusionne comme précédemment.
- `how="left"` garde **tous nos films**, même ceux absents de TMDB.
- `left_on="tconst"` et `right_on="imdb_id"` relient les deux tableaux par l'identifiant IMDb.
- `.drop(columns=["imdb_id"])` supprime la colonne en double.

**Point d'attention.** Les numéros de ligne vont maintenant jusqu'à **6 450**, alors que nous avions 6 447 films. Cela veut dire que **quelques films apparaissent deux fois dans TMDB** et ont été dupliqués par la fusion. Pour corriger, nous pourrions ajouter `films_tmdb = films_tmdb.drop_duplicates(subset="imdb_id")` avant la fusion.

In [17]:
films_francais[films_francais["poster_path"].isna() & films_francais["title"].isna()]

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path
3991,tt14418234,movie,State of Siege: Temple Attack,0,2021.0,110.0,"Action,Drama,Thriller",7.5,10603,NaN,en,NaN
4097,tt15176374,movie,Unpaarvayil,0,2025.0,117.0,Thriller,9.0,10057,NaN,ta,NaN
4164,tt15576290,movie,Do Aur Do Pyaar,0,2024.0,140.0,"Comedy,Drama,Romance",6.4,12432,NaN,NaN,NaN
4696,tt21626774,movie,Sector 36,0,2024.0,123.0,"Crime,Drama,Thriller",7.1,27629,NaN,en,NaN
5084,tt27866781,movie,Kaalidhar Laapata,0,2025.0,109.0,"Comedy,Drama",7.5,10361,NaN,NaN,NaN
5106,tt28089700,movie,Raid 2,0,2025.0,145.0,"Crime,Drama,Thriller",6.6,37238,NaN,NaN,NaN
5139,tt28717622,movie,Bomb,0,2025.0,139.0,"Comedy,Drama",6.1,15027,NaN,NaN,NaN
5236,tt30798612,movie,Crazxy,0,2025.0,93.0,"Drama,Thriller",6.6,18139,NaN,hi,NaN
5243,tt30889755,movie,The Sabarmati Report,0,2024.0,127.0,"Crime,Drama,Thriller",6.4,25877,NaN,NaN,NaN
5261,tt31349844,movie,Runner,0,2026.0,98.0,"Action,Comedy,Thriller",6.2,20348,NaN,NaN,NaN


Repérer les films les plus « pauvres » en informations.

**Explication du code :**
- `.isna()` renvoie Vrai quand la case est vide.
- `&` veut dire « et » : nous voulons les films **sans affiche ET sans titre français**.

**17 films**, presque tous **indiens et très récents** (2024 à 2026). Ce sont de bons candidats à retirer.

In [18]:
films_francais[films_francais["original_language"].isna() & films_francais["title"].isna()]

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path
4164,tt15576290,movie,Do Aur Do Pyaar,0,2024.0,140.0,"Comedy,Drama,Romance",6.4,12432,NaN,NaN,NaN
5084,tt27866781,movie,Kaalidhar Laapata,0,2025.0,109.0,"Comedy,Drama",7.5,10361,NaN,NaN,NaN
5106,tt28089700,movie,Raid 2,0,2025.0,145.0,"Crime,Drama,Thriller",6.6,37238,NaN,NaN,NaN
5139,tt28717622,movie,Bomb,0,2025.0,139.0,"Comedy,Drama",6.1,15027,NaN,NaN,NaN
5243,tt30889755,movie,The Sabarmati Report,0,2024.0,127.0,"Crime,Drama,Thriller",6.4,25877,NaN,NaN,NaN
5261,tt31349844,movie,Runner,0,2026.0,98.0,"Action,Comedy,Thriller",6.2,20348,NaN,NaN,NaN
5433,tt34936199,movie,Daagi,0,2025.0,142.0,"Action,Romance",6.1,11317,NaN,NaN,NaN
5535,tt37809949,movie,Dulhaniya Le Aaeegi,0,2026.0,127.0,Comedy,7.4,10600,NaN,NaN,NaN
5562,tt38525492,movie,Bhagwat Chapter One: Raakshas,0,2025.0,127.0,"Crime,Thriller",6.8,10509,NaN,NaN,NaN
5577,tt39122441,movie,Jockey,0,2026.0,147.0,"Action,Adventure,Drama",7.7,14752,NaN,NaN,NaN


Même idée avec la langue originale : un film sans langue dans TMDB est souvent un film **absent de TMDB**.

**Explication du code :**
- `films_francais["original_language"].isna()` : pas de langue originale.
- `films_francais["title"].isna()` : pas de titre français.
- `&` garde les films qui remplissent **les deux** conditions.

**12 films**, tous déjà présents dans la liste précédente. Ce sont les films que nous connaissons le moins bien.

In [19]:
films_francais[~films_francais["poster_path"].isna() & films_francais["title"].isna() & films_francais["original_language"].str.contains("en")]

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path
380,tt0081748,movie,Where the Buffalo Roam,0,1980.0,99.0,"Biography,Comedy",6.5,13409,NaN,en,/vRMeb4LN416PGvSgG0n063bc5ez.jpg
447,tt0083869,movie,Eating Raoul,0,1982.0,90.0,"Comedy,Crime",6.8,10137,NaN,en,/1B37isseQEN81emH0nSzuMWOCyl.jpg
462,tt0084509,movie,The Plague Dogs,0,1982.0,103.0,"Adventure,Animation,Drama",7.7,10327,NaN,en,/97Z56DEjHNNQXOPW9s9rL15khjM.jpg
513,tt0086373,movie,Strange Brew,0,1983.0,90.0,"Comedy,Crime,Sci-Fi",6.6,20448,NaN,en,/rScplcvRHX7o7XK88SFb5mddo8y.jpg
820,tt0095188,movie,Funny Farm,0,1988.0,101.0,Comedy,6.2,25900,NaN,en,/tW6LGDEZrJpLbvEGkHk51MyQqJC.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...
5707,tt4669186,movie,Kevin Hart: What Now?,0,2016.0,96.0,"Comedy,Documentary",6.0,10364,NaN,en,/lFg4tx0jAYkgrcm1bPBlM0FbQvH.jpg
5720,tt4714568,movie,The Young Offenders,0,2016.0,83.0,"Adventure,Comedy,Crime",7.1,14022,NaN,en,/qpRz7J6Jw4M7Oa02dCRpot7gNO2.jpg
5932,tt5929750,movie,Beatriz at Dinner,0,2017.0,82.0,"Comedy,Drama",6.0,11499,NaN,en,/ypvlGBvm6p0EIvKCzJw0OqsuzK5.jpg
6097,tt6952960,movie,The Kindergarten Teacher,0,2018.0,96.0,"Drama,Thriller",6.7,19075,NaN,en,/pwavkdZb2xNRh5Y9MHhkZJ2UaHT.jpg


Trouver les films que nous pouvons **garder malgré l'absence de titre français**. Un film en **anglais** peut être projeté en France avec son titre original, qui reste compréhensible. S'il a en plus une **affiche**, il est utilisable dans l'application.

**Explication du code :**
- `~films_francais["poster_path"].isna()` : le `~` veut dire « **non** ». Donc « l'affiche n'est **pas** vide », c'est-à-dire que le film **a une affiche** ;
- `films_francais["title"].isna()` : **pas de titre français** ;
- `original_language.str.contains("en")` : la langue originale est l'**anglais** ;
- les trois conditions sont reliées par `&` (« et »).

**108 films** répondent à ces conditions (*Where the Buffalo Roam*, *Strange Brew*, *Beatriz at Dinner*...).

In [20]:
films_francais[(~films_francais["poster_path"].isna() & films_francais["original_language"].str.contains("en")) | ~films_francais["title"].isna()]

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path
0,tt0035423,movie,Kate & Leopold,0,2001.0,118.0,"Comedy,Fantasy,Romance",6.4,95169,Kate et Léopold,en,/mUvikzKJJSg9khrVdxK8kg3TMHA.jpg
1,tt0064106,movie,The Butcher,0,1970.0,93.0,"Drama,Mystery,Thriller",7.3,10675,Le boucher,fr,/An2gkXn5TATag9Adul2Zbq5Ndrv.jpg
2,tt0064177,movie,Colossus: The Forbin Project,0,1970.0,100.0,"Sci-Fi,Thriller",7.1,12058,Le cerveau d'acier,en,/kmVDlTzERxSvBrqtQurjKf1zEy1.jpg
3,tt0065134,movie,Two Mules for Sister Sara,0,1970.0,116.0,"Adventure,Drama,Romance",7.0,33505,Sierra torride,en,/RbuNHMlrw7M2w1R3hZMAPwh6dG.jpg
4,tt0065143,movie,The Bird with the Crystal Plumage,0,1970.0,96.0,"Drama,Horror,Mystery",7.1,27767,L'oiseau au plumage de cristal,it,/dpALRkwodEIT7TGX9SqTgwVNn0i.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...
6445,tt9873892,movie,They Cloned Tyrone,0,2023.0,122.0,"Comedy,Mystery,Sci-Fi",6.7,49479,Ils ont cloné Tyrone,en,/omSa3oafsV7i5776FLFoOBI2Ho9.jpg
6447,tt9893250,movie,I Care a Lot,0,2020.0,118.0,"Comedy,Crime,Drama",6.4,161411,I Care a Lot,en,/gKnhEsjNefpKnUdAkn7INzIFLSu.jpg
6448,tt9894470,movie,VFW,0,2019.0,92.0,"Action,Crime,Horror",6.1,10551,VFW,en,/AnVD7Gn14uOTQhcc5xYZGQ9DRvS.jpg
6449,tt9900782,movie,Kaithi,0,2019.0,145.0,"Action,Crime,Thriller",8.4,53416,Kaithi,ta,/hOF9CgPsy9aLr5GJEBESC8MEXFy.jpg


Écrire la règle qui décide **quels films nous gardons** : un film est gardé **s'il a un titre français**, **OU** s'il a une affiche **ET** une langue originale anglaise.

**Explication du code :**
- Le `|` veut dire « **ou** ».
- Le `&` veut dire « **et** ».
- Les parenthèses indiquent ce qui va ensemble, comme en mathématiques.

**6 092 films** passent ce filtre. Cette cellule sert seulement à **voir** le tableau ; il n'est pas encore enregistré dans une variable.

In [21]:
selection = films_francais[~films_francais["title"].isna() | (~films_francais["poster_path"].isna() & films_francais["original_language"].str.contains("en"))]

selection.info()

<class 'pandas.DataFrame'>
Index: 6092 entries, 0 to 6450
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tconst             6092 non-null   str    
 1   titleType          6092 non-null   str    
 2   primaryTitle       6092 non-null   str    
 3   isAdult            6092 non-null   int64  
 4   startYear          6092 non-null   float64
 5   runtimeMinutes     6092 non-null   float64
 6   genres             6092 non-null   str    
 7   averageRating      6092 non-null   float64
 8   numVotes           6092 non-null   int64  
 9   title              5984 non-null   str    
 10  original_language  5912 non-null   str    
 11  poster_path        5796 non-null   str    
dtypes: float64(3), int64(2), str(7)
memory usage: 618.7 KB


Enregistrer le filtre dans une nouvelle variable, `selection`.

**Explication du code :**
- `selection = ...` applique **exactement la même règle** que la cellule précédente (les deux parties du « ou » sont juste écrites dans l'autre ordre).
- `selection.info()` affiche un résumé du tableau : le nombre de lignes et, pour chaque colonne, le nombre de cases **remplies** (*Non-Null Count*).

Nous obtenons :
- **6 092 films** dans la sélection ;
- `title` rempli pour **5 984 films** : il manque donc le titre français pour 108 films, les films anglophones gardés grâce à la règle ;
- `poster_path` rempli pour 5 796 films : une partie des films avec titre français n'a pas d'affiche.

In [22]:
selection["sans_titre"] = selection["title"].isna()

selection["title"] = selection["title"].fillna(selection["primaryTitle"])

selection[selection["sans_titre"]]

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path,sans_titre
380,tt0081748,movie,Where the Buffalo Roam,0,1980.0,99.0,"Biography,Comedy",6.5,13409,Where the Buffalo Roam,en,/vRMeb4LN416PGvSgG0n063bc5ez.jpg,True
447,tt0083869,movie,Eating Raoul,0,1982.0,90.0,"Comedy,Crime",6.8,10137,Eating Raoul,en,/1B37isseQEN81emH0nSzuMWOCyl.jpg,True
462,tt0084509,movie,The Plague Dogs,0,1982.0,103.0,"Adventure,Animation,Drama",7.7,10327,The Plague Dogs,en,/97Z56DEjHNNQXOPW9s9rL15khjM.jpg,True
513,tt0086373,movie,Strange Brew,0,1983.0,90.0,"Comedy,Crime,Sci-Fi",6.6,20448,Strange Brew,en,/rScplcvRHX7o7XK88SFb5mddo8y.jpg,True
820,tt0095188,movie,Funny Farm,0,1988.0,101.0,Comedy,6.2,25900,Funny Farm,en,/tW6LGDEZrJpLbvEGkHk51MyQqJC.jpg,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5707,tt4669186,movie,Kevin Hart: What Now?,0,2016.0,96.0,"Comedy,Documentary",6.0,10364,Kevin Hart: What Now?,en,/lFg4tx0jAYkgrcm1bPBlM0FbQvH.jpg,True
5720,tt4714568,movie,The Young Offenders,0,2016.0,83.0,"Adventure,Comedy,Crime",7.1,14022,The Young Offenders,en,/qpRz7J6Jw4M7Oa02dCRpot7gNO2.jpg,True
5932,tt5929750,movie,Beatriz at Dinner,0,2017.0,82.0,"Comedy,Drama",6.0,11499,Beatriz at Dinner,en,/ypvlGBvm6p0EIvKCzJw0OqsuzK5.jpg,True
6097,tt6952960,movie,The Kindergarten Teacher,0,2018.0,96.0,"Drama,Thriller",6.7,19075,The Kindergarten Teacher,en,/pwavkdZb2xNRh5Y9MHhkZJ2UaHT.jpg,True


Que **chaque film ait un titre à afficher**. Pour les 108 films sans titre français, nous utilisons le **titre original**, et nous gardons une trace de ces films pour savoir plus tard lesquels ont été complétés.

**Explication du code :**
- `selection["sans_titre"] = selection["title"].isna()` crée une nouvelle colonne qui vaut Vrai pour les films **sans titre français**. Il faut la créer **avant** de remplir les titres, sinon nous perdrions l'information.
- `fillna(selection["primaryTitle"])` remplit les cases vides de `title` avec le titre original du **même film**.
- `selection[selection["sans_titre"]]` affiche les films qui ont été complétés.

Les **108 films** complétés apparaissent bien, avec `sans_titre` à `True`.

In [23]:

# détection de novueaux doublons
doublons = selection[selection["tconst"].duplicated(keep=False)]
print(selection["tconst"].duplicated().sum(), "doublons")
doublons

3 doublons


,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path,sans_titre
4401,tt1756855,movie,Coyote vs. Acme,0,2026.0,103.0,"Adventure,Animation,Comedy",7.3,51477,Coyote vs. Acme,en,NaN,False
4402,tt1756855,movie,Coyote vs. Acme,0,2026.0,103.0,"Adventure,Animation,Comedy",7.3,51477,Coyote vs. Acme,en,NaN,False
4839,tt23468450,movie,Longlegs,0,2024.0,101.0,"Crime,Horror,Mystery",6.5,237152,Longlegs,en,NaN,False
4840,tt23468450,movie,Longlegs,0,2024.0,101.0,"Crime,Horror,Mystery",6.5,237152,Longlegs,en,/3wVnnLzX5eObFWzrsy5vJGpMCCI.jpg,False
5050,tt27441481,movie,Coolie,0,2025.0,168.0,"Action,Crime,Drama",6.0,25450,Coolie,ta,/3GUVDQ0arZcZzxRAVfyAxijTKEe.jpg,False
5051,tt27441481,movie,Coolie,0,2025.0,168.0,"Action,Crime,Drama",6.0,25450,Coolie,ta,/hDntDhxW0uKsCQRaUfcwCyYUXKY.jpg,False


In [24]:
# suppression des doublons
selection = selection.drop([4193, 4603, 4087]).reset_index(drop=True)

In [25]:
selection.info()

<class 'pandas.DataFrame'>
RangeIndex: 6089 entries, 0 to 6088
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tconst             6089 non-null   str    
 1   titleType          6089 non-null   str    
 2   primaryTitle       6089 non-null   str    
 3   isAdult            6089 non-null   int64  
 4   startYear          6089 non-null   float64
 5   runtimeMinutes     6089 non-null   float64
 6   genres             6089 non-null   str    
 7   averageRating      6089 non-null   float64
 8   numVotes           6089 non-null   int64  
 9   title              6089 non-null   str    
 10  original_language  5909 non-null   str    
 11  poster_path        5793 non-null   str    
 12  sans_titre         6089 non-null   bool   
dtypes: bool(1), float64(3), int64(2), str(7)
memory usage: 576.9 KB


Vérifier que tous les titres sont remplis.

**Explication du code :**
- `selection.info()` affiche à nouveau le résumé du tableau.

La colonne `title` est maintenant remplie pour les **6 089 films**. La nouvelle colonne `sans_titre` (de type `bool`, c'est-à-dire Vrai ou Faux) a bien été ajoutée : il y a désormais **13 colonnes**.

In [26]:
selection.to_csv("../data/films_selection.csv", index=False)

Sauvegarder le travail pour les étapes suivantes du projet.

**Explication du code :**
- `to_csv("../data/films_selection.csv", ...)` enregistre la sélection dans le dossier `data/`.
- `index=False` évite d'ajouter une colonne avec les numéros de ligne.

Le fichier `films_selection.csv` contient **6 092 films**, chacun avec un titre à afficher. La sélection **reste à affiner** (par exemple les films sans affiche, ou les quelques doublons venus de TMDB).